# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohamedRamadan164/FlyRank_ML_internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane: Refresh / Content Opportunity Scoring.** This notebook has two kinds of cells:
- **Starter-sample cells** (marked ✅) run on `data/raw/content_refresh_anonymized.csv`, already executed for real below — no network needed.
- **Warehouse cells** (marked 🌐, requires `HF_TOKEN`) query the full v20260703 release on Hugging Face. These were written against the documented schema but **not executed by the AI assistant that drafted them** — it has no network access to Hugging Face and never had your token. Every real output number in a 🌐 cell is marked `[[FILL IN AFTER YOU RUN]]`. You have a working token, so run these yourself in Colab and replace the placeholders with your actual numbers before this counts as done.

## 1. Question

*The research question and the decision it supports.*

**Question.** Which content items in a client's catalog are worth a human review this month — and specifically, which look like they should be *refreshed* (declining or underperforming their position's usual CTR), which should be left alone (*protect* — growing, don't touch), and which are ambiguous and need a closer look (*monitor*)?

**Decision it supports.** A content/SEO editor working a fixed-size review queue each month needs a ranked, reason-coded shortlist instead of scrolling an entire catalog. This is **decision-support**, not automation — the model ranks and explains, a human decides.

**Continuity.** This is the direct extension of `w02` (task framing) → `w03` (data contract, leakage trap) → `w04` (the `review_for_refresh` baseline rule, two signal verdicts: staleness MIXED, CTR-vs-position CONFIRMED) → `w05` (logistic regression beats the baseline at every K tested on a client-grouped split). The capstone's job is to re-run that same validated chain at full-warehouse scale and turn it into a shipped, ranked action queue with reason codes.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** `FlyRank/internship-warehouse`, build `v20260703` (~81.8M rows total). **Tables used:** `dim_content` (519,606 rows, one per content item — static/dimensional attributes, hashed keys: `content_hash_id`, `client_hash_id`, `content_type`, `word_count`, `char_count`, `search_volume`, `competition`, `cpc`, `main_intent`, `backlinks`, `content_created_date`, `content_updated_date`, `last_optimized_date`, `is_published`, `is_deleted`) joined to `fact_content_daily_performance` (78,835,655 rows, `report_date x client x content`, month-partitioned).

**Date window:** a single **mid-panel month** (`month=2026-03`) for iteration and label construction — **never** `fact_content_daily_performance_sample`, which is the panel's final, sealed month (June 2026) and would be the natural outcome window of any past→future label built on it.

**Excluded, and why:**
- `fact_content_query_90d` (2,414,248 rows) — its 90-day window overlaps the month boundary; mixing it into a single-month analysis without first aligning the windows would be a silent leak, not a feature. Deferred to a future iteration once that alignment is done properly.
- `is_deleted = TRUE` rows — a deleted page isn't a candidate for a refresh review.
- Any GA4-derived column where `ga4_data_available` is not TRUE — those rows are zero-filled placeholders, not real zero engagement (per `flyrank-data` skill).
- `client_hash_id` / `content_hash_id` are pseudonyms used only for joining, grouping, and the client-grouped split — never features.

**Public-safe:** no client names, domains, raw queries, or credentials appear anywhere below or in the deployed paper — only pseudonymous hash IDs and aggregate statistics, per the FlyRank Internship Data Use Terms.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label (proxy, not an observed future outcome).** `is_declining_this_month` = 1 if a content item's clicks fell from the first half of the analysis month to the second half, else 0 — the same proxy construction validated in `w03`'s leakage trap, now applied warehouse-scale.

**Features (knowable before the decision moment, day 15 of the month):**
1. `word_count`, `content_type`, `backlinks`, `search_volume`, `competition` — static, from `dim_content`, knowable at any time.
2. `content_age_days` (derived from `content_created_date`) — static.
3. `days_since_last_update` (derived from `content_updated_date` vs the decision date) — static as of the decision moment.
4. `gsc_avg_position_first_half`, `clicks_first_half`, `impressions_first_half` — aggregated from `fact_content_daily_performance`, March 1–15 only.
5. `ctr_gap` — the same CTR-vs-position benchmark logic validated in `w04` (Signal B, CONFIRMED), refit on the training clients only.

**Baseline.** The `w04` rule, re-expressed with reason codes: `score = stale_flag x visible_flag x ctr_gap x impressions_first_half`, reason code `stale_ctr_underperform`, action `review_for_refresh`.

**Validation design.** Client-grouped split (`client_hash_id`), same as `w05` — every content item from a given client lands entirely in train or entirely in test, so the model is evaluated on clients it has never seen, not on held-out rows from clients it already learned from.

**Leakage checks.** (1) Peek real column names before trusting anything — schema names are confirmed live, not assumed. (2) Exclude `clicks_second_half`/`impressions_second_half` — they define the label. (3) Exclude any GA4 column when `ga4_data_available` is not TRUE. (4) Exclude `fact_content_query_90d` entirely (window overlap, see Section 2). (5) Assert client sets in train/test don't intersect before training anything.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

### 4a. Validated prototype (starter sample, already real — ✅ executed below)

Before scaling to the full warehouse, the same rule/model/split logic was already validated for real on the 30,000-row starter sample in `w04`/`w05`. That result is reproduced here as a sanity check the warehouse run should roughly track:

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/MohamedRamadan164/FlyRank_ML_internship"
REPO_DIR = "FlyRank_ML_internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")  # starter sample, used by every ✅ cell below
print("starter sample rows:", len(df))

# reproduced directly from w05's real, executed output -- not re-derived here to keep this cell fast
prototype_table = pd.DataFrame([
    {"K": 20, "base_rate": 0.517, "baseline_rule_P@K": 0.40, "logistic_regression_P@K": 0.65, "random_forest_P@K": 0.60},
    {"K": 50, "base_rate": 0.517, "baseline_rule_P@K": 0.40, "logistic_regression_P@K": 0.62, "random_forest_P@K": 0.56},
    {"K": 100, "base_rate": 0.517, "baseline_rule_P@K": 0.46, "logistic_regression_P@K": 0.60, "random_forest_P@K": 0.59},
])
print("Prototype result (starter sample, 30,000 rows, client-grouped split, from w05):")
prototype_table


starter sample rows: 30000
Prototype result (starter sample, 30,000 rows, client-grouped split, from w05):


,K,base_rate,baseline_rule_P@K,logistic_regression_P@K,random_forest_P@K
0,20,0.517,0.40,0.65,0.60
1,50,0.517,0.40,0.62,0.56
2,100,0.517,0.46,0.60,0.59


### 4b. Full-warehouse result (🌐 — run this yourself, requires `HF_TOKEN`)

This mirrors `w03`'s connection pattern exactly: peek real columns first, then build the same pipeline as `w05` at warehouse scale.

In [2]:
%pip -q install duckdb
import duckdb

con = duckdb.connect()
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    import getpass
    HF_TOKEN = getpass.getpass("HF_TOKEN (not shown, not saved to disk): ")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
DIM_CONTENT = f"read_parquet('{BASE}/dim_content.parquet')"
FACT_MAR = f"read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')"

# Peek real columns -- do not trust names below until confirmed against this output.
print(con.sql(f"SELECT * FROM {DIM_CONTENT} LIMIT 3").df().columns.tolist())
print(con.sql(f"SELECT * FROM {FACT_MAR} LIMIT 3").df().columns.tolist())


['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'mo

In [4]:
features = con.sql(f"""
    WITH daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            gsc_clicks,
            gsc_impressions,
            gsc_avg_position
        FROM {FACT_MAR}
    )

    SELECT
        d.client_hash_id,
        d.content_hash_id,

        c.word_count,
        c.backlinks,
        c.search_volume,
        c.competition,

        DATE_DIFF(
            'day',
            c.content_created_date,
            DATE '2026-03-15'
        ) AS content_age_days,

        DATE_DIFF(
            'day',
            c.content_updated_date,
            DATE '2026-03-15'
        ) AS days_since_last_update,

        AVG(
            CASE
                WHEN d.report_date <= DATE '2026-03-15'
                THEN d.gsc_avg_position
            END
        ) AS gsc_avg_position_first_half,

        SUM(
            CASE
                WHEN d.report_date <= DATE '2026-03-15'
                THEN d.gsc_clicks
                ELSE 0
            END
        ) AS clicks_first_half,

        SUM(
            CASE
                WHEN d.report_date <= DATE '2026-03-15'
                THEN d.gsc_impressions
                ELSE 0
            END
        ) AS impressions_first_half,

        SUM(
            CASE
                WHEN d.report_date > DATE '2026-03-15'
                THEN d.gsc_clicks
                ELSE 0
            END
        ) AS clicks_second_half

    FROM daily d

    JOIN {DIM_CONTENT} c
        ON c.content_hash_id = d.content_hash_id

    WHERE c.is_deleted IS NOT TRUE

    GROUP BY
        d.client_hash_id,
        d.content_hash_id,
        c.word_count,
        c.backlinks,
        c.search_volume,
        c.competition,
        c.content_created_date,
        c.content_updated_date
""").df()

features["is_declining_this_month"] = (
    features["clicks_second_half"] < features["clicks_first_half"]
).astype(int)

print("shape:", features.shape)
print(
    "declining rate:",
    features["is_declining_this_month"].mean()
)

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

shape: (324947, 13)
declining rate: 0.08918685200971235


,client_hash_id,content_hash_id,word_count,backlinks,search_volume,competition,content_age_days,days_since_last_update,gsc_avg_position_first_half,clicks_first_half,impressions_first_half,clicks_second_half,is_declining_this_month
0,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,2855,0,10,0.00,31,-106,3.737399,1.0,219.0,0.0,1
1,client_62f4a7e64f5e0096,content_d49a012dcb924e31,2993,0,10,0.00,31,-106,4.520919,0.0,246.0,0.0,0
2,client_62f4a7e64f5e0096,content_4dc944b7d0b65ecc,3698,0,10,0.62,31,-106,4.449490,0.0,70.0,0.0,0
3,client_62f4a7e64f5e0096,content_4a1ca0fa5c177e0c,2995,0,10,0.00,31,-106,5.238095,0.0,11.0,0.0,0
4,client_62f4a7e64f5e0096,content_7a7d3c7aa7cdfc5c,3373,96,10,0.00,31,-106,14.285714,0.0,12.0,0.0,0


In [5]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

num_feats = ["word_count", "backlinks", "search_volume", "competition", "content_age_days",
             "days_since_last_update", "gsc_avg_position_first_half", "clicks_first_half", "impressions_first_half"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(features, groups=features["client_hash_id"]))
train, test = features.iloc[train_idx], features.iloc[test_idx]
assert not (set(train["client_hash_id"]) & set(test["client_hash_id"]))

pre = ColumnTransformer([("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_feats)])
logit = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))])
logit.fit(train[num_feats], train["is_declining_this_month"])
score = logit.predict_proba(test[num_feats])[:, 1]

yte = test["is_declining_this_month"].values
warehouse_table = pd.DataFrame([
    {"K": k, "base_rate": round(yte.mean(), 3), "logistic_regression_P@K": round(precision_at_k(score, yte, k), 3)}
    for k in [20, 50, 100]
])
print(f"Test: {len(test)} rows, {test['client_hash_id'].nunique()} held-out clients")
warehouse_table
# [[FILL IN AFTER YOU RUN]]: does this track the starter-sample prototype in 4a? Report both, don't only keep the flattering one.


Test: 52578 rows, 14 held-out clients


,K,base_rate,logistic_regression_P@K
0,20,0.071,0.40
1,50,0.071,0.50
2,100,0.071,0.48


## 5. Limitations

*What this work cannot claim.*

- **The label is a proxy, not an observed future outcome.** `is_declining_this_month` compares two halves of the *same* month, computed after the fact — it is not a genuine past→future forecast. A stronger version would use a prior window's features to predict a later, separate window's outcome.
- **Uneven client history.** `dim_clients.gsc_data_start`/`ga4_data_start` vary per client, so one global calendar month does not mean equal information for every client in it. Some clients may have little usable history in March 2026 at all.
- **Single-month snapshot.** One mid-panel month may not represent seasonal variation across the full ~17-month panel; a result that holds in March isn't guaranteed to hold in a different month.
- **Small buckets mislead.** `w04` already found this directly — the 181+ day freshness tier had only n=174 rows and gave a result that reversed the overall trend. Any bucket this small in the warehouse cut deserves the same suspicion.
- **No causal claim.** Nothing here shows that acting on a recommendation *causes* a traffic or ranking change — there is no experimental (before/after, treatment/control) design in this data. This is decision-support, not proof.
- **Not a model of Google's algorithm.** The label is a proxy computed from FlyRank's own panel, not search-engine internals — the model never sees, and cannot infer, anything about how Google actually ranks pages.

In [6]:
print("Limitations logged: proxy label, uneven client history, single-month snapshot,")
print("small-bucket risk (see w04's n=174 case), no causal claim, no algorithm claim.")


Limitations logged: proxy label, uneven client history, single-month snapshot,
small-bucket risk (see w04's n=174 case), no causal claim, no algorithm claim.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Three action labels, one reason code family, matching the baseline's `stale_ctr_underperform` logic extended with a growth check:

| Action | Condition | Reason code |
|---|---|---|
| `refresh` | stale (≥90d) AND visible AND under-capturing CTR for its position tier | `stale_ctr_underperform` |
| `protect` | growing (clicks rose first-half → second-half) AND already well-positioned | `growing_leave_alone` |
| `monitor` | none of the above clearly apply — ambiguous, worth a light periodic check | `ambiguous_watch` |

The queue is written to `work/outputs/refresh_action_queue.csv` (starter-sample version, regenerated on every run — stays out of git per the CI leak-guard, same rule as `w04`'s CSV).

In [7]:
# Starter-sample version of the action playbook -- real, executed.
df2 = df.copy()
valid2 = df2[df2["avg_position"] > 0]
bench2 = valid2.groupby("position_tier")["clicks_90d"].sum() / valid2.groupby("position_tier")["impressions_90d"].sum() * 100
df2["expected_ctr"] = df2["position_tier"].map(bench2)
df2["ctr_gap"] = (df2["expected_ctr"] - df2["ctr"]).clip(lower=0).where(df2["avg_position"] > 0, 0)
vis_thr2 = df2["impressions_90d"].median()
stale2 = df2["days_since_last_update"] >= 90
visible2 = df2["impressions_90d"] >= vis_thr2
growing2 = df2["trend_direction"] == "up"
well_positioned2 = df2["position_tier"].isin(["top_3", "page_1"])

conditions = [
    stale2 & visible2 & (df2["ctr_gap"] > 0),
    growing2 & well_positioned2,
]
choices = ["refresh", "protect"]
import numpy as np
df2["action"] = np.select(conditions, choices, default="monitor")
df2["reason_code"] = np.select(conditions, ["stale_ctr_underperform", "growing_leave_alone"], default="ambiguous_watch")
df2["score"] = (stale2.astype(int) * visible2.astype(int) * df2["ctr_gap"] * df2["impressions_90d"])

os.makedirs("work/outputs", exist_ok=True)
queue = df2.sort_values("score", ascending=False)
out_cols = ["content_id", "client_id", "action", "reason_code", "score", "ctr", "avg_position", "days_since_last_update", "trend_direction"]
queue[out_cols].to_csv("work/outputs/refresh_action_queue.csv", index=False)

print(queue["action"].value_counts())
print("\nwrote work/outputs/refresh_action_queue.csv")
queue[out_cols].head(10)


action
monitor    24470
refresh     4129
protect     1401
Name: count, dtype: int64

wrote work/outputs/refresh_action_queue.csv


,content_id,client_id,action,reason_code,score,ctr,avg_position,days_since_last_update,trend_direction
6653,content_5fe46e04994d,client_4e07408562,refresh,stale_ctr_underperform,108887.742905,0.14,4.2,104,down
3394,content_36ff89c8214e,client_19581e27de,refresh,stale_ctr_underperform,88624.627778,0.05,7.3,104,stable
7445,content_c8e9d6ab9013,client_19581e27de,refresh,stale_ctr_underperform,73104.852519,0.00,9.7,104,down
3331,content_4a6607efcb46,client_6208ef0f77,refresh,stale_ctr_underperform,61286.156110,0.01,2.2,104,up
26531,content_cb112fce36be,client_19581e27de,refresh,stale_ctr_underperform,58983.222991,0.16,5.6,104,down
26474,content_a7427266c305,client_19581e27de,refresh,stale_ctr_underperform,48331.742956,0.11,5.7,104,stable
3070,content_91652435f57a,client_19581e27de,refresh,stale_ctr_underperform,46332.761922,0.06,7.8,104,stable
9193,content_c1fe78bc4e37,client_19581e27de,refresh,stale_ctr_underperform,42940.995820,0.03,7.5,104,down
5621,content_97a86caf3a3d,client_19581e27de,refresh,stale_ctr_underperform,41395.403221,0.07,6.4,104,down
4708,content_b115f7c74779,client_19581e27de,refresh,stale_ctr_underperform,39550.048957,0.03,8.0,104,up


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Three real charts, generated from the starter-sample data (figures below); the warehouse-scale versions use the same plotting code once `features`/`warehouse_table` from Section 4b are filled in.

In [8]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("work/outputs/figures", exist_ok=True)

# Figure 1: staleness vs decline rate
g1 = df.groupby("freshness_tier").agg(n=("content_id", "size"), decline_rate=("trend_direction", lambda s: (s == "down").mean()))
g1 = g1.reindex(["0-30", "31-90", "91-180", "181+"])
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(g1.index, g1["decline_rate"] * 100, color="#4C72B0")
ax.set_ylabel("Decline rate (%)"); ax.set_xlabel("Freshness tier")
ax.set_title("Staleness vs decline rate (MIXED signal, w04)")
for i, (idx, row) in enumerate(g1.iterrows()):
    ax.text(i, row["decline_rate"] * 100 + 1, f"n={int(row['n'])}", ha="center", fontsize=8)
plt.tight_layout(); plt.savefig("work/outputs/figures/staleness_vs_decline.png", dpi=150); plt.close()

# Figure 2: CTR vs position
ctr2 = valid2.groupby("position_tier").apply(lambda s: 100 * s["clicks_90d"].sum() / s["impressions_90d"].sum())
ctr2 = ctr2.reindex(["top_3", "page_1", "striking", "page_3_5", "deep"])
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(ctr2.index, ctr2.values, color="#55A868")
ax.set_ylabel("Weighted CTR (%)"); ax.set_xlabel("Position tier")
ax.set_title("CTR vs position (CONFIRMED signal, w04)")
plt.tight_layout(); plt.savefig("work/outputs/figures/ctr_vs_position.png", dpi=150); plt.close()

# Figure 3: model vs baseline (from w05's real, executed result)
K = [20, 50, 100]; baseline_p = [0.40, 0.40, 0.46]; logit_p = [0.65, 0.62, 0.60]; rf_p = [0.60, 0.56, 0.59]
x = np.arange(len(K)); width = 0.25
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(x - width, baseline_p, width, label="Baseline rule (w04)", color="#C44E52")
ax.bar(x, logit_p, width, label="Logistic Regression", color="#4C72B0")
ax.bar(x + width, rf_p, width, label="Random Forest", color="#8172B2")
ax.axhline(0.517, color="gray", linestyle="--", linewidth=1, label="Base rate (0.52)")
ax.set_xticks(x); ax.set_xticklabels([f"K={k}" for k in K])
ax.set_ylabel("Precision@K"); ax.set_title("Model vs baseline (starter-sample prototype)")
ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("work/outputs/figures/model_vs_baseline.png", dpi=150); plt.close()

print("Saved 3 figures to work/outputs/figures/")


/tmp/ipykernel_2250/1894912915.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ctr2 = valid2.groupby("position_tier").apply(lambda s: 100 * s["clicks_90d"].sum() / s["impressions_90d"].sum())


Saved 3 figures to work/outputs/figures/


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
